In [ ]:
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List

# SWOT 분석 결과를 위한 Pydantic 모델 정의
class SWOTKeywords(BaseModel):
    strength_keyword: str
    weakness_keyword: str
    opportunity_keyword: str
    threat_keyword: str

class SWOTResponse(BaseModel):
    keywords: List[SWOTKeywords]

# LLM 및 출력 파서 초기화
llm = ChatOpenAI(temperature=0.7, model_name="gpt-4o")
parser = PydanticOutputParser(pydantic_object=SWOTResponse)

# 프롬프트 템플릿 정의
template = """...{review_text}...{format_instructions}"""
prompt = PromptTemplate(
    template=template,
    input_variables=["review_text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# 분석 체인 구성
swot_chain = LLMChain(llm=llm, prompt=prompt)

def run_swot_analysis(text):
    """
    리뷰 텍스트를 기반으로 SWOT 키워드를 구조화하여 반환합니다.
    """
    parsed = swot_chain | parser
    return parsed.invoke({"review_text": text})
